## 06 Agent with Memory

Agents **do not** remember previous conversations by default. In this notebook, we'll address the issue by equiping the agent with "memory", which will help it preserve the entire context of the conversation.

<div align="center">
<img src="images/06_agents_with_memory.png" width="450" heigh="300" alt="Agent with Memory"/>
</div>

In [1]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from typing import Literal, Union

import langchain
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)
console = Console()

print(f"Using langchain version: {langchain.__version__}")

load_dotenv(override=True)

Using langchain version: 1.2.14


True

In [2]:
# let's create our LLM and agent first.

# using OpenAI as LLM
openai_llm = init_chat_model(
    "gpt-4o-mini",  # replace this with any supported OpenAI model name
    model_provider="openai",
    temperature=0.7,  # make it creative!
)

agent = create_agent(
    model=openai_llm,
    system_prompt="You are a helpful assistant",
)

In [7]:
# helper function to ask question & get a response from agent
def ask_agent(agent, query: str, config=None) -> str:
    response = agent.invoke(
        {"messages": {"role": "user", "content": query}}, config=config
    )
    # response = agent.invoke({"messages": {"role": "user", "content": query}})
    return response["messages"][-1].content

As a first step, let's understand the issue of lack of memory.

In [8]:
queries = [
    "Hi, My name is Manish. Who are you?",
    "What is the capital of India?",
    "What is my name?",
]

for query in queries:
    response = ask_agent(agent, query)
    console.print(f"[green]**Query:[/green]** {query}")
    console.print(Markdown(f"**Agent Response:** {response}"))
    console.print("-+" * 35 + "\n")

**Query:** Hi, My name is Manish. Who are you?

Agent Response: Hi Manish! I'm an AI assistant here to help you with information, answer your questions, and assist
you with a variety of topics. How can I assist you today?

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

**Query:** What is the capital of India?

Agent Response: The capital of India is New Delhi.

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

**Query:** What is my name?

Agent Response: I'm sorry, but I don't have access to your personal information, including your name. How can I    
assist you today?

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

Oops! The agent has no clue what my name is, even though I told it very explicitly in the very first query. This is to be expected as the agent has no memory by default, so it cannot remember any conversation history.

* We can fix this by adding _short-term-memory_ to our agent. 
* Memory will _preserve_ all messages exchanged with the agent so far, so the agent now has context or previous requests and responses.

To add memory, we use an instance of the `InMemorySaver` class from the `langgraph.checkpoint.memory` package. **NOTE** it is `langgraph`, not `langchain`. Here is how we add memory to our agent above.

```python
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=openai_llm,
    system_prompt="You are a helpful assistant",
    # add the short-term memory like this
    checkpointer=InMemorySaver(),
)    
```

We will have to `invoke()` the agent with it's own _thread_, by passing in a configurable, like this.

```python
config = {"configurable": {"thread_id":"1024"}} # value of thread_id can be any unique number
response = agent.invoke(agent.invoke({"messages": ...}, config=config)
```

Now let's try it on the list of messages above & see if our agent maintains context.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

agent2 = create_agent(
    model=openai_llm,
    system_prompt="You are a helpful agent",
    checkpointer=InMemorySaver(),
)

In [10]:
config = {"configurable": {"thread_id": "123"}}

for query in queries:
    response = ask_agent(agent2, query, config)
    console.print(f"[green]**Query:[/green]** {query}")
    console.print(Markdown(f"**Agent Response:** {response}"))
    console.print("-+" * 35 + "\n")

**Query:** Hi, My name is Manish. Who are you?

Agent Response: Hi Manish! I'm an AI language model here to assist you with any questions or information you might 
need. How can I help you today?

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

**Query:** What is the capital of India?

Agent Response: The capital of India is New Delhi. If you have any more questions or need further information, feel
free to ask!

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

**Query:** What is my name?

Agent Response: Your name is Manish. How can I assist you further?

-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+

Perfect! This time it remembered that my name is "Manish" - all previous requests and responses are saved in the instance of `InMemorySaver()` short-term memory and sent to agent as part of it's context.

#### Runtime Context
LangChain's `create_agent()` runs on LangGraph's runtime under the hood.

LangGraph exposes a _Runtime_ object with the following information:
* _Context_: additional (other than messages) static information like `user_id`, `db connections`, or other dependencies you'd like to pass to the agent on each invocation.
* _Store_: a `BaseStore` instance used for _long-term_ memory
* _Stream writer_: an object used for streaming information via the _custom_ stream mode

The _context_ is actually useful in the tools rather than to the LLM itself. You can access _runtime information_ in your tools as well as via custom agent middleware. For example, suppose you have an agent that queries databases, then you would want to pass in the database connection (& maybe the schema) to all your tools, so they can easily query the database (or even perform CRUD operations, if that makes sense).

Here is how you would pass in runtime context to an agent (and via the agent to the tools!).

1. First you define the runtime schema - you would typically use an instance of a `dataclass` like this:

    ```python
    from dataclasses import dataclass

    @dataclass
    class RuntimeContext:
        # define any number of data members
        name: str
        db: SQLDatabase
        ... etc. etc.
    ```

    You can also use an instance of `TypedDict` to pass in context information - this class would be familiar to LangGraph users, where we use it to define the state of the graph.

    ```python
    from typing import TypedDict

    class RuntimeContext(TypedDict):
        # define any number of data members
        name: str
        db: SQLDatabase
        ... etc. etc.
    ```

2. Next, you provide the schema to the agent via the `context_schema` parameter of the `create_agent()` function. 

    ```python
    from langchain.agents import create_agent

    agent = create_agent(
        model="openai:gpt-40-mini',
        system_prompt="you are a helpful agent",
        tools=[execute_query],
        # here we tell the agent the "type" of the context
        # if you are familiar with LangGraph, this is the same as
        # instantiating the StateGraph
        context_schema=RuntimeContext,
        ....
    )
    ```
3. In any tool function (decorated by the `@tool` decorator), you can access this context using the `langgraph.runtime.get_runtime(...)` function as follows:

    ```python
    from langchain_core.tools import tool
    from langgraph.runtime import get_runtime

    @tool
    def execute_query(sql: str) -> str:
        # get the context defined in create_agent call
        runtime = get_runtime(RuntimeContext)
        # now you can access the fields of the context
        # if RuntimeContext is a dataclass then use
        db = runtime.context.db
        # OR if RuntimeContext is a TypedDict instance then use
        db = runtime.context["db"]
        name = runtime.context.name
        # ... etc
        ...
    ```

4. Finally, when you invoke the agent, you should provide an instance of the context in the context variable, as follows:

    ```python
    # when streaming
    for chunk in agent.stream(
        {"messages" : user_query},
        # here provide the initialized context
        context=RuntimeContext(name="Biblo Baggins", db=SQLDatabase(....)),
        stream_mode="values"
    ):
        chunk["messages"][-1].pretty_print()

    # when invoking
    response = agent.invoke(
        {"messaghes" : user_query},
        context=RuntimeContext(name="Biblo Baggins", db=SQLDatabase(....)),
    )
    ```
Now let's see how we use the database in an example below. Here we are using a local SQLite database connection to a `Chinook` database.

In [6]:
from langchain_community.utilities import SQLDatabase

# connect to our database -> in path db/chinook.db
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")

# Step 1: create a class derived from TypedDict that defines the runtime-context

# from dataclasses import dataclass

# # define the runtime context for our agent
# @dataclass
# class RuntimeContext:
#     db: SQLDatabase

from typing import TypedDict


# define the runtime context for our agent
class RuntimeContext(TypedDict):
    db: SQLDatabase

In [7]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """execute query provided by user
    Args:
        query (str): SQL query to execute
    Returns:
        str: result of the query execution or error message
    """
    runtime = get_runtime()
    # context is initialized when we initialize the agent - see code below
    # if runtime was a dataclass I would use following syntax
    # db: SQLDatabase = runtime.context.db
    # since it id derived from TypedDict, I'll use the following syntax
    db: SQLDatabase = runtime.context["db"]
    try:
        result = db.run(query)
    except Exception as e:
        return f"Error occurred while executing SQL query: {e}"
    return str(result)

In [8]:
SYSTEM_PROMPT = """You are a careful SQLite Analyst.

Rules:
- Always think step-by-step
- When you need data, call the tool 'execute_sql' with ONE select query
- Read-only only; NO INSERT/UPDATE/DELETE/DROP/CREATE/REPLACE/TRUNCATE/ALTER
- Limit to 5 rows at the output, unless the user explicitly asks for more
- If the tool returns "Error:", revise the SQL and try again
- Prefer explicit column list, avoid SELECT *
"""

Create the agent. Add model, tools, prompt and runtime context.

**NOTE:** while we expect the agent to translate text-to-SQL, we are not providing the agent with the database schema, so the agent is going to make mistakes at first. But due to instructions in the System prompt, the agent will try & self-correct until it generates the SQL.

In [9]:
from langchain.agents import create_agent

# we are using the short-cut method to specify the model the
# agent will use - providing a name, such as "openai:gpt-5" is
# enough - you can always use your favourite LLM, such as
# "anthropic:claude-haiku-4-5" or ""google_genai:gemini-2.5-flash

agent = create_agent(
    # configure the appropriate API KEY in the .env file!!
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

**NOTE the following:**
- The agent does not have access to the database schema
- The agent may make mistakes, but by self-correction is expected to correct those
- We'll invoke the agent with `agent.stream()` and display all messages with pretty printing so you can see how the agent is working
- Agent does not remember the schema

In [10]:
# utility function
def ask_agent(agent, query: str):
    for step in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        # this is where the context is initialized for our agent
        context=RuntimeContext(db=db),
        stream_mode="values",
    ):
        # messages accumulate in a scratch-buffer - we display the last one
        step["messages"][-1].pretty_print()

In [11]:
query = "This is Frank Harris. What was the total on my last invoice?"
ask_agent(agent, query)

================================ Human Message =================================

This is Frank Harris. What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_0px893tVCVIr4VJZj5I6weCE)
 Call ID: call_0px893tVCVIr4VJZj5I6weCE
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table' AND lower(name) LIKE '%invoice%';
================================= Tool Message =================================
Name: execute_sql

[('invoices', 'CREATE TABLE "invoices"\r\n(\r\n    [InvoiceId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,\r\n    [CustomerId] INTEGER  NOT NULL,\r\n    [InvoiceDate] DATETIME  NOT NULL,\r\n    [BillingAddress] NVARCHAR(70),\r\n    [BillingCity] NVARCHAR(40),\r\n    [BillingState] NVARCHAR(40),\r\n    [BillingCountry]...'), ('invoice_items', 'CREATE TABLE "invoice_items"\r\n(\r\n    [InvoiceLineId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,\r\n    [InvoiceId] INTEG

So the agent is able to eventually figure out the total of the invoice = `$5.94`. What if we ask a follow up question like `What were the titles`, where we imply the titles for which the invoice amounted to `$5.94`.

In [12]:
query = "What were the titles?"
ask_agent(agent, query)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Could you clarify which titles you mean? For example, titles of books, movies, job titles, etc. If you want me to query the database, please tell me the table/column (and any filters or date range). If you’re not sure, I can first look for any tables that have a “title” column.


Clearly our agent has gottem  horribly confused as it does not have the context of the previous query. So, it is correctly asking us to clarify `which titles do you mean??`

### Short Term Memory
Let's solve this problem by adding short term memory to our agent.

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

In [22]:
agent = create_agent(
    # configure the appropriate API KEY in the .env file!!
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    # here is how we provide short-term memory
    checkpointer=InMemorySaver(),
)

In [ ]:
# utility function
def ask_agent(agent, query: str):
    steps = []
    for step in agent.stream(
        {"messages": [{"role": "user", "content": query}]},
        # you now MUST add this parameter to ensure we track state on a thread
        {"configurable": {"thread_id": 1024}},
        # this is where the context is initialized for our agent
        context=RuntimeContext(db=db),
        stream_mode="values",
    ):
        # messages accumulate in a scratch-buffer - we display the last one
        step["messages"][-1].pretty_print()
        steps.append(step)
    return steps

In [24]:
query = "This is Frank Harris. What was the total on my last invoice?"
steps = ask_agent(agent, query)

================================ Human Message =================================

This is Frank Harris. What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_Zrv2pKNRM7z8sF3RQaZDF2Rs)
 Call ID: call_Zrv2pKNRM7z8sF3RQaZDF2Rs
  Args:
    query: SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name LIMIT 100;
================================= Tool Message =================================
Name: execute_sql

[('albums', 'table'), ('artists', 'table'), ('customers', 'table'), ('employees', 'table'), ('genres', 'table'), ('invoice_items', 'table'), ('invoices', 'table'), ('media_types', 'table'), ('playlist_track', 'table'), ('playlists', 'table'), ('sqlite_sequence', 'table'), ('sqlite_stat1', 'table'), ('tracks', 'table')]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_B584CqO2YFdrE9DVd5WhyTjh)
 Call 

Ok! We got the same response. Now how about the follow up question??

In [25]:
query = "What were the titles?"
steps = ask_agent(agent, query)

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_hGVSQYHu3F1nXx3kEBsqTaqe)
 Call ID: call_hGVSQYHu3F1nXx3kEBsqTaqe
  Args:
    query: SELECT ii.InvoiceLineId, t.Name AS TrackTitle
FROM invoice_items ii
JOIN tracks t ON t.TrackId = ii.TrackId
WHERE ii.InvoiceId = 374
ORDER BY ii.InvoiceLineId;
================================= Tool Message =================================
Name: execute_sql

[(2021, 'Holier Than Thou'), (2022, 'Through The Never'), (2023, 'My Friend Of Misery'), (2024, 'The Wait'), (2025, 'Blitzkrieg'), (2026, 'So What')]
================================== Ai Message ==================================

Here are the track titles on your last invoice (Invoice 374):
- Holier Than Thou
- Through The Never
- My Friend Of Misery
- The Wait
- Blitzkrieg
- So What

Want prices or artists too?


Wow! The agent remembered the previous context and is now able to fetch the titles on that `$5.94` invoice!!

### Conclusion
Memory is quite essential when building agents, especially if you are going to use a conversational interface with an agent. This example illustrated how you can use short term memory with agents using `langgraph.checkpoint.memory.InMemorySaver` class.